<!-- 5 Tank Recycle Case -->

3 Tank Recycle Scenario

In [ ]:
include("3 Tank Recycle WAS.jl")
include("MPC Model 3tk Recycle WAS.jl")

create_mpc_model (generic function with 1 method)

In [15]:
##############################
# ASSIGN WEIGHTS 
##############################
#alpha_tk = assign_weights()
alpha_tk = nothing
##############################
# CREATE MPC MODEL
##############################
mpc_model = create_mpc_model(alpha_tk)

println(alpha_tk)

nothing


In [ ]:
##############################
# RUN MPC
##############################
using JuMP

n_steps = Int(nt/dt)+1     

#STATE PROFILES
Vprofile = zeros(n_steps,NI)  
Ssprofile = zeros(n_steps, 1)
Xbprofile = zeros(n_steps, 1)
Soprofile = zeros(n_steps, 1)

#CONCENTRATION PROFILES
#Algebraic variable 
Xb_flowprofile = zeros(n_steps, O)
Ss_flowprofile = zeros(n_steps, O)
#State variable
Xb_tkprofile = zeros(n_steps, NI)
Ss_tkprofile = zeros(n_steps, NI)

#INPUT PROFILES
F_profile = zeros(n_steps-1, O)
Klaprofile = zeros(n_steps-1, 1)
RASprofile = zeros(n_steps-1, 1)

#STATE INITIAL CONDITIONS
Vprofile[1, :] = V0
Ssprofile[1] = Ss0
Xbprofile[1] = Xb0
Soprofile[1] = So0

Xb_tkprofile[1, :] = Xb_tk0
Ss_tkprofile[1, :] = Ss_tk0


for i = 2:n_steps

    fmax = fmax_vec[:, i]
    Bd = Bd_vec[:, i]
    Ss_in_max = Ss_in_vec[i]

    # FIX INITIAL CONDITIONS IN MODEL AND RUN 
    fix.(mpc_model[:V][:, 1], V0, force=true)
    fix.(mpc_model[:Ss][1, 1], Ss0, force=true)
    fix.(mpc_model[:Xb][1, 1], Xb0, force=true)
    fix.(mpc_model[:So][1, 1], So0, force=true)

    fix.(mpc_model[:Xb_tk][1, :, 1], Xb_tk0, force=true)
    fix.(mpc_model[:Ss_tk][1, :, 1], Ss_tk0, force=true)

    #Boundary codition 
    fix.(mpc_model[:Xb_flow][:, 1, :], Xb_in, force=true)
    
    #fix.(mpc_model[:RAS][:], RAS0, force=true)
    #fix.(mpc_model[:Kla][:], Kla0, force=true)
    
    #=
    for t in 1:npred 
        fix.(mpc_model[:f][:,t], f0, force=true)
    end
    =#
    
    # FIX DISTURBANCES IN MODEL
    fix.(mpc_model[:fmax], fmax, force=true)
    fix.(mpc_model[:Bd], Bd, force=true)
    fix.(mpc_model[:Ss_flow][:, 1, :], Ss_in_max, force=true)

    # UPDATE STARTING GUESSES
    Xb_flow_guess = repeat(reshape(Xb_flow0, 1, O, 1), Col, 1, npred)
    Ss_flow_guess = repeat(reshape(Ss_flow0, 1, O, 1), Col, 1, npred)

    set_start_value.(mpc_model[:Xb_flow], Xb_flow_guess)
    set_start_value.(mpc_model[:Ss_flow], Ss_flow_guess)
    

    # SOLVE MODEL 
    optimize!(mpc_model)
    status = JuMP.termination_status(mpc_model)

    if status != MOI.LOCALLY_SOLVED && status != MOI.OPTIMAL
        println("Model did not converge to global or local optima")
    else
        println("Model solved successfully")
    end
    
    println(status)
    

    # SAVE RESULTS FOR STATES, FROM 2ND TIME STEP (FIRST TIME STEP IS INITIAL CONDITION)
    V0 = [value(mpc_model[:V][ni, 2]) for ni in 1:NI]
    Ss0 = value(mpc_model[:Ss][1, 2])
    Xb0 = value(mpc_model[:Xb][1, 2])
    So0 = value(mpc_model[:So][1, 2])

    Xb_tk0 = [value.(mpc_model[:Xb_tk][1, ni, 2]) for ni in 1:NI]
    Ss_tk0 = [value.(mpc_model[:Ss_tk][1, ni, 2]) for ni in 1:NI]

    if i==2
        Xb_flowprofile[1, :] = [value.(mpc_model[:Xb_flow][1, o, 1]) for o in 1:O]
        Ss_flowprofile[1, :] = [value.(mpc_model[:Ss_flow][1, o, 1]) for o in 1:O]
        Xb_flow0 = [value.(mpc_model[:Xb_flow][1, o, 2]) for o in 1:O]
        Ss_flow0 = [value.(mpc_model[:Ss_flow][1, o, 2]) for o in 1:O]
    else
        Xb_flow0 = [value.(mpc_model[:Xb_flow][1, o, 2]) for o in 1:O]
        Ss_flow0 = [value.(mpc_model[:Ss_flow][1, o, 2]) for o in 1:O]
    end

    #SAVE RESULTS FOR INPUTS, FROM 1ST TIME STEP 
    f0 = value.(mpc_model[:f][:, 1])
    Kla0 = value(mpc_model[:Kla][1])
    RAS0 = value(mpc_model[:RAS][1])

    #UPDATE PROFILES FOR STATES
    Vprofile[i, :] = V0 
    Ssprofile[i] = Ss0 
    Xbprofile[i] = Xb0 
    Soprofile[i] = So0
    Xb_tkprofile[i, :] = Xb_tk0
    Ss_tkprofile[i, :] = Ss_tk0
    
    Xb_flowprofile[i, :] = Xb_flow0
    Ss_flowprofile[i, :] = Ss_flow0
    
    #UPDATE PROFILES FOR INPUTS
    Klaprofile[i-1] = Kla0 
    F_profile[i-1, :] = f0
    RASprofile[i-1] = RAS0



end 

Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved successfully
LOCALLY_SOLVED
Model solved suc